In [2]:
%%capture
# ── Install Unsloth + training stack ─────────────────────────────────────────
# Unsloth must be installed FIRST — it patches torch internals at import time.
# The [colab-new] extra auto-selects the right CUDA / xformers build for this GPU.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# xformers build depends on the torch version already installed by Colab
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton

# Evaluation + visualisation helpers
!pip install Pillow wandb codebleu sentence_transformers



In [ ]:
import os, subprocess

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  OPTION A — RunPod                                                      │
# │  Run this cell OR the local cell below. Not both.                       │
# └─────────────────────────────────────────────────────────────────────────┘

# ── Pull latest code from GitHub ─────────────────────────────────────────────
REPO_URL = "https://github.com/kasmello/ChoAI.git"
REPO_DIR = "/workspace/ChoAI"



os.chdir(REPO_DIR)

# ── Paths ─────────────────────────────────────────────────────────────────────
# Upload ProcessedData/ and Training/ to the pod after it starts, or store
# them on a RunPod Network Volume mounted at /workspace.
DATA_DIR = "/workspace/ChoAI"

# ── Secrets ───────────────────────────────────────────────────────────────────
# Set WANDB_API_KEY in the pod config before starting:
# RunPod dashboard → Edit Pod → Environment Variables
# They'll already be in os.environ here — nothing else needed.
print(f"Working directory : {os.getcwd()}")
print(f"DATA_DIR          : {DATA_DIR}")
print(f"WANDB_API_KEY set : {'WANDB_API_KEY' in os.environ}")

FileNotFoundError: [Errno 2] No such file or directory: '/workspace/ChoAI'

In [3]:
import os
from dotenv import load_dotenv

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  OPTION B — Local machine                                               │
# │  Run this cell OR the Colab cell above. Not both.                       │
# └─────────────────────────────────────────────────────────────────────────┘

# ── Load secrets from .env ────────────────────────────────────────────────────
load_dotenv()

DATA_DIR = "."

print(f"Working directory : {os.getcwd()}")
print(f"DATA_DIR          : {os.path.abspath(DATA_DIR)}")
print(f"WANDB_API_KEY set : {'WANDB_API_KEY' in os.environ}")

Working directory : /content
DATA_DIR          : /content
WANDB_API_KEY set : False


In [7]:
import os
print(os.getcwd())

/Users/karmel/Desktop/ChoAI


In [5]:
import requests
import json
import csv
import pandas as pd
import numpy as np
import os
import wandb
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import matplotlib.image as mpimg
from io import StringIO, BytesIO
from tqdm import tqdm
import PIL
import gc

from datasets import load_dataset, Dataset
from codebleu import calc_codebleu

import warnings
warnings.filterwarnings("ignore")

# ── Secrets + paths ───────────────────────────────────────────────────────────
# WANDB_API_KEY, HF_TOKEN, and DATA_DIR are set by whichever config cell you
# ran above (Colab or local) — nothing Colab-specific in this cell.
os.environ["WANDB_PROJECT"]   = "ChoAI"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

# ── Load assets ───────────────────────────────────────────────────────────────
logo_img = mpimg.imread("Media/horizontal-logo.png")

with open(f"{DATA_DIR}/Training/final_file.json", "r") as json_file:
    data = json.load(json_file)

df = pd.read_csv(f"{DATA_DIR}/ProcessedData/Joined_DF.csv")
df["action_time"] = pd.to_datetime(df["action_time"], format="ISO8601")
df["Date"]        = pd.to_datetime(df["Date"])

# ── Custom fonts (shipped in repo under font/) ────────────────────────────────
manjari_bold_path    = "font/Manjari-Bold.ttf"
manjari_regular_path = "font/Manjari-Regular.ttf"
manjari_thin_path    = "font/Manjari-thin.ttf"

manjari_bold    = fm.FontProperties(fname=manjari_bold_path)
manjari_regular = fm.FontProperties(fname=manjari_regular_path)
manjari_thin    = fm.FontProperties(fname=manjari_thin_path)

# ── Brand colours ─────────────────────────────────────────────────────────────
dark   = "#3B3365"
medium = "#8F81DD"
light  = "#BCADFF"
peach  = "#FFEEDD"

df

/Users/karmel/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/karmel/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


FileNotFoundError: [Errno 2] No such file or directory: '/workspace/ChoAI/Training/final_file.json'

In [4]:
for entry in range(len(data)):

  sample_df = pd.read_csv(data[entry]['input'])
  columns = str(list(sample_df.columns))
  data[entry]['columns'] = columns



In [ ]:
from nltk.translate.bleu_score import sentence_bleu

def compute_bleu(predictions, references, language='python'):
    """
    Computes the BLEU score.

    Args:
        predictions (list of str): Model-generated outputs.
        references (list of str): Ground truth outputs.
        language (str): Programming language of the code (e.g., "python", "java").

    Returns:
        dict: Dictionary containing the CodeBLEU score.
    """
    bleu_score = sentence_bleu([predictions], references)
    return {"codebleu": bleu_score}

def compute_metrics(eval_pred):
    """
    Wrapper for HuggingFace Trainer to compute CodeBLEU.

    Args:
        eval_pred: A tuple of (predictions, labels).

    Returns:
        dict: Dictionary of evaluation metrics.
    """
    predictions, labels = eval_pred

    # Convert model token IDs back to strings
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute CodeBLEU
    return compute_bleu(decoded_predictions, decoded_labels, language="python")

In [5]:
from unsloth import FastLanguageModel
import torch

# ── Sequence length ───────────────────────────────────────────────────────────
# 2048 covers virtually all generated matplotlib snippets.
# Increase only if you see truncated outputs.
max_seq_length = 2048
dtype    = None   # auto-detect: float16 for T4/V100, bfloat16 for A100+

# ── Models available to benchmark ────────────────────────────────────────────
# All are 4-bit quantised Unsloth variants — fits on a free Colab T4 (15 GB).
# Swap MODEL_KEY to run a different experiment; everything else stays the same.
AVAILABLE_MODELS = {
    "llama":   "unsloth/Llama-3.2-3B-Instruct",             # original baseline
    "mistral": "unsloth/mistral-7b-instruct-v0.3",           # strong general model
    "gemma":   "unsloth/gemma-3-4b-it-bnb-4bit",             # Google Gemma 3 — very efficient
    "qwen":    "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit", # code-specialised Qwen
}

# ── Pick the model for this run ───────────────────────────────────────────────
# Change this to "gemma" or "qwen" to run a different experiment.
MODEL_KEY  = "llama"
model_name = AVAILABLE_MODELS[MODEL_KEY]
lr         = 2e-4
use_4bit   = True
lora_alpha = 16

# ── Alpaca prompt template ────────────────────────────────────────────────────
# Instruction = natural language query
# Input       = column names of the DataFrame the model will query
# Response    = executable matplotlib Python code
alpaca_prompt = """
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}
"""

# ── Build HuggingFace Dataset from training JSON ──────────────────────────────
dataset = Dataset.from_dict({
    "instruction": [entry["instruction"].lower() for entry in data],
    "input":       [entry["columns"] for entry in data],
    "output":      [entry["output"] for entry in data],
})

# 80 / 10 / 10 split — fixed seeds make results reproducible across runs
split_dataset         = dataset.train_test_split(test_size=0.2, seed=89)
validation_test_split = split_dataset['test'].train_test_split(test_size=0.5, seed=42)

train = split_dataset['train']
val   = validation_test_split['train']
test  = validation_test_split['test']

print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")


NotImplementedError: Unsloth currently only works on NVIDIA, AMD and Intel GPUs.

In [ ]:
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from transformers import TrainingArguments, EarlyStoppingCallback

# ── Load base model + attach LoRA adapters ────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_name,
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = use_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = lora_alpha,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = False,
    loftq_config   = None,
)

# ── Format each example as an Alpaca-style prompt ────────────────────────────
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for instruction, input, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

train = train.map(formatting_prompts_func, batched=True)
val   = val.map(formatting_prompts_func, batched=True)
test  = test.map(formatting_prompts_func, batched=True)

model_filename = f"{model_name.replace('unsloth/', '')}_{lr}_{lora_alpha}"
if use_4bit:
    model_filename += "_4bit"

callbacks = [EarlyStoppingCallback(early_stopping_patience=4)]

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train,
    eval_dataset       = val,
    dataset_text_field = "text",
    max_seq_length     = max_seq_length,
    dataset_num_proc   = 2,
    packing            = False,
    callbacks          = callbacks,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        max_steps                   = 200,
        learning_rate               = lr,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        evaluation_strategy         = "steps",
        logging_strategy            = "steps",
        save_strategy               = "steps",
        eval_steps                  = 25,
        save_steps                  = 25,
        logging_steps               = 1,
        output_dir                  = "outputs",
        report_to                   = "wandb",
        save_total_limit            = 3,
        run_name                    = model_filename,
        load_best_model_at_end      = True,
    ),
)

run = wandb.init(project="ChoAI", name=model_filename)

try:
    trainer.train(resume_from_checkpoint=False)
except ValueError as e:
    print(f"Training ended early: {e}")

wandb.log({
    "model_name": model_name.replace("unsloth/", ""),
    "lr":         lr,
    "use_4bit":   use_4bit,
    "lora_alpha": lora_alpha,
})

# ── Save adapter locally ───────────────────────────────────────────────────────
model.save_pretrained(model_filename)
tokenizer.save_pretrained(model_filename)
print(f"Adapter saved to ./{model_filename}")

# ── Upload adapter to Modal Volume ────────────────────────────────────────────
# The adapter saved above needs to be uploaded to a Modal Volume so the
# serving app (modal_app.py) can load it at inference time.
#
# After downloading the adapter from Colab to your local machine, run:
#   modal volume put choai-models ./{model_filename}/ /models/{model_filename}/
#
# See modal_app.py for the full serving setup.
print(f"TODO: upload ./{model_filename} to Modal Volume (see modal_app.py)")

wandb.finish()

In [ ]:


trainer_EVAL = SFTTrainer(
  model = model,
  tokenizer = tokenizer,
  eval_dataset = test,
  dataset_text_field = "text",
  # data_collator=data_collator,
  max_seq_length = max_seq_length,
  dataset_num_proc = 2,
  packing = False, # Can make training 5x faster for short sequences.
  callbacks=callbacks,
  args = TrainingArguments(

      seed = 3407,
      evaluation_strategy="steps",
      logging_strategy="steps",
      save_strategy="steps",
      eval_steps = 10,
      save_steps = 10,
      logging_steps = 1,
      output_dir = "results",
      report_to = "wandb",
      save_total_limit = 3,
      run_name = model_filename,
      load_best_model_at_end = True,
      do_eval = True
  ),

)


# try:
  # artifact = run.use_artifact(f'kasmello/ChoAI/{run.id}', type='model')
  # print('Artifact found!')
  # artifact_dir = artifact.download()
results = trainer_EVAL.evaluate()
wandb.log({"Testing Evaluation Loss": results['eval_loss']})
# except wandb.CommError:
#   print('Artifact not found - training from scratch')
#   trainer.train()
# wandb.log({
#     'model_name': model_name.replace('unsloth/',''),
#     'lr': lr,
#     'use_4bit': use_4bit,
#     'lora_alpha': lora_alpha
# })
# model.save_pretrained(model_filename) # Local saving
# tokenizer.save_pretrained(model_filename)
# wandb.finish()

Map (num_proc=2):   0%|          | 0/179 [00:00<?, ? examples/s]

In [ ]:
trainer_EVAL

{'eval_loss': 0.0770912617444992,
 'eval_model_preparation_time': 0.0089,
 'eval_runtime': 31.0676,
 'eval_samples_per_second': 5.762,
 'eval_steps_per_second': 0.74}

In [ ]:
import torch
import gc
torch.cuda.empty_cache()
del trainer
gc.collect()


9085